In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import cartopy.crs as ccrs
import cmocean
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from OceanDataStore import OceanDataCatalog 
from nemo_cookbook import NEMODataTree  


In [2]:
# Open model and config files
catalog = OceanDataCatalog(catalog_name="noc-model-stac")
catalog.search(collection='noc-npd-era5', standard_name='sea_surface_temperature')
ds1 = catalog.open_dataset(id=catalog.Items[2].id,
                          start_datetime='1990-01',
                          end_datetime='2024-12')
catalog.search(collection='noc-npd-era5', item_name='domain_cfg')
config = catalog.open_dataset(id=catalog.Items[1].id)

# Merge into a data tree
datasets = {"parent": {"domain": config, "gridT": ds1}}
dt_global = NEMODataTree.from_datasets(datasets = datasets)

# Clip to North Atlantic 
bbox = (-85.0, 0.0, 0.0, 80.0)
dt_clipped = dt_global.clip_grid(grid="gridT", bbox=bbox)

# Add lat and lon as co-ordinates
dt = dt_clipped.add_geoindex(grid="gridT")

# Correct compatibility error 
dt['gridT']['tmaskutil'] = dt['gridT']['tmaskutil'].astype(bool)


# Compute areas
areas = dt.cell_area(grid="gridT", dim="k")

# Convert to dataset
ds = (dt['gridT']).dataset




            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1y
              Title: eORCA1 ERA5v1 NPD T1y Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics annual mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2025-07-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1m
              Title: eORCA1 ERA5v1 NPD T1m Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics monthly mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2025-07-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d
              Title: eORCA025 ERA5v1 NPD T1y_3d Icechunk repository
              Description: Icechunk repository containing eORCA025 ERA5v1 

In [3]:
ds

<xarray.DatasetView> Size: 1GB
Dimensions:                (time_counter: 35, j: 482, i: 341, axis_nbounds: 2,
                            k: 75)
Coordinates:
  * time_counter           (time_counter) datetime64[ns] 280B 1990-07-02T12:0...
  * j                      (j) int64 4kB 685 686 687 688 ... 1163 1164 1165 1166
  * i                      (i) int64 3kB 809 810 811 812 ... 1146 1147 1148 1149
  * k                      (k) int64 600B 1 2 3 4 5 6 7 ... 69 70 71 72 73 74 75
    time_centered          (time_counter) datetime64[ns] 280B dask.array<chunksize=(1,), meta=np.ndarray>
  * gphit                  (j, i) float64 1MB 0.0 0.0 0.0 ... 79.41 79.3 79.2
  * glamt                  (j, i) float64 1MB -85.0 -84.75 -84.5 ... 48.54 48.79
Dimensions without coordinates: axis_nbounds
Data variables: (12/57)
    berg_latent_heat_flux  (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
    evs                    (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
    ficeberg               (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
    hfds                   (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
    fsitherm               (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
    hfempds                (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
    ...                     ...
    e1t                    (j, i) float64 1MB dask.array<chunksize=(482, 341), meta=np.ndarray>
    e2t                    (j, i) float64 1MB dask.array<chunksize=(482, 341), meta=np.ndarray>
    top_level              (j, i) int32 657kB dask.array<chunksize=(482, 341), meta=np.ndarray>
    bottom_level           (j, i) int32 657kB dask.array<chunksize=(482, 341), meta=np.ndarray>
    tmask                  (k, j, i) float64 99MB nan 1.0 1.0 ... nan nan nan
    tmaskutil              (j, i) bool 164kB True True True ... True True True
Indexes:
  ┌ gphit    NDPointIndex (SklearnGeoBallTreeAdapter)
  └ glamt
Attributes:
    nftype:   None
    iperio:   False

In [4]:
weighted_mean = dt.masked_statistic(grid='gridT', var='sos_abs', lon_poly = [-80.0, 0.0, 0.0, -80.0, -80.0], lat_poly = [60.0, 60.0, 80.0, 80.0, 60.0], statistic="weighted_mean", dims=["i", "j"])


In [6]:
ds1['tos_con']

<xarray.DataArray 'tos_con' (time_counter: 35, y: 331, x: 360)> Size: 17MB
dask.array<getitem, shape=(35, 331, 360), dtype=float32, chunksize=(1, 331, 360), chunktype=numpy.ndarray>
Coordinates:
  * time_counter   (time_counter) datetime64[ns] 280B 1990-07-02T12:00:00 ......
    nav_lat        (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    nav_lon        (y, x) float64 953kB dask.array<chunksize=(331, 360), meta=np.ndarray>
    time_centered  (time_counter) datetime64[ns] 280B dask.array<chunksize=(1,), meta=np.ndarray>
Dimensions without coordinates: y, x
Attributes:
    standard_name:       sea_surface_temperature
    long_name:           sea_surface_conservative_temperature
    units:               degC
    online_operation:    average
    interval_operation:  3600 s
    interval_write:      1 yr
    cell_methods:        time: mean (interval: 3600 s)